# Reproducing Hickman et al. (2026) from dandiset 000774

**Paper.** Hickman, Hughes, Sahai, Miles and Denman (2026), "Neural population dynamics of direct
electrical stimulation of neocortex", Cell Reports 45, 117420, doi:10.1016/j.celrep.2026.117420.

**Dataset.** DANDI:000774, version `0.260520.1753` (doi:10.48324/dandi.000774/0.260520.1753),
streamed from the DANDI API. This is the version the paper cites in its key resources table, and the
only published version of the dandiset; the draft has the same ten assets.

**Text the claims were written from.** The NIH author manuscript (PMC13404931) with the STAR Methods
and Document S1 (legends of Figures S1 to S12); Table S1 was not in the text read
(`claims/000774/10.1016-j.celrep.2026.117420.yaml`). Method details the claims file does not give
are taken from the paper and cited where used; where the paper is silent, from its code
(github.com/denmanlab/estim_populations at c4aaa608), as the claims file does.

**What the dandiset holds.** Spike times, Kilosort templates, CCF coordinates and regions of the
units, and the table of electrical pulses, for ten acute recordings. The raw AP band, the LFP band
and the behaviour are not in it, as the paper says ("Raw data (LFP band, AP band, and NiDAQ binary
files) are not included in the NWB"), so the evoked-potential claims (Figures 1 to 3, S1, S2), the
claims that compare units with the evoked-potential boundaries, and the behaviour claims (Figure 7,
S10) are blocked.

| Claim | Tested here |
|---|---|
| fig1l-saturation-amplitude, figs1b-saturation-polarity | blocked, raw AP band not in the dandiset |
| fig2g-extent-increases, fig2g-extent-sublinear, fig2-extent-polarity, fig2h-probe-identity, figs2gh-magnitude-linear | blocked, raw AP band |
| fig3c-volume, fig3e-ventral-compression, fig3fg-anisotropy-strength, fig3h-com-displacement | blocked, raw AP band |
| fig4f-responders-within-ep, fig4i-proportion-within-ep, fig5i-fs-within-ep | blocked, need the evoked-potential boundaries |
| fig4d-latency-trough, figs3cd-gmm-window | tested |
| fig4h-extent-constant, fig4h-distance-decreases, fig4h-ks-distinct, fig4j-drp-increases, figs3e-early-windows | tested |
| fig5bc-amplitude-clusters, fig5ef-polarity-clusters | tested |
| fig5-fsrs-proportion, fig5j-fs-closer, figs6b-fsrs-early-firing | tested |
| fig6c-modulation-cortex, fig6d-hva-subcortical, fig6e-circuit-responders, fig6f-no-propagating-wave | tested |
| figs9-state-drp, fig6g-dimensionality-collapse | tested |
| fig7d to fig7j, figs10c, figs10e, no-learning-across-sessions (10 claims) | blocked, behaviour not in the dandiset |

Every random step has its own seeded generator. The analysis takes about 12 minutes, most of it
the ZETA tests (9 minutes on four cores).

In [ ]:
import os
for _v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ.setdefault(_v, "1")
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.cluster.hierarchy import linkage, fcluster, cophenet
from scipy.spatial.distance import pdist
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm, AnovaRM
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
import estim

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
T0 = time.time()

ASSETS = estim.list_assets()
DATA = {}
for a in ASSETS:
    DATA[a["session"]] = estim.load_session(a)
print(f"loaded {len(DATA)} files in {time.time() - T0:.0f} s")

In [ ]:
# distances from the dataset's coordinates, and, as a variant, from the paper's code coordinates
CODE_COORDS = {s: estim.code_coords(s) for s in estim.PAPER_SESSIONS}
for s in estim.PAPER_SESSIONS:
    u, tr, trains, meta = DATA[s]
    contact_xyz = tr[["neg_ap", "neg_dv", "neg_ml"]].iloc[0].to_numpy()
    u["distance"], u["norm_distance"] = estim.signed_distances(u, contact_xyz)
    u["distance_code"], u["norm_distance_code"] = estim.code_distances(
        u, CODE_COORDS[s], tr.contact_negative.iloc[0])
    u["keep"] = (u.group == "good") & ~u.unstable
    u["sub_cort"] = u.region.map(estim.classify_region)
    u["region_group"] = u.region.map(estim.map_region_to_group)

## Data check

In [ ]:
rows = []
for s, (u, tr, trains, meta) in DATA.items():
    probes = "".join(sorted(u.probe.unique()))
    rows.append(dict(session=s, subject=meta["subject"], probes=probes, units=len(u),
                     good=int((u.group == "good").sum()),
                     good_stable=int(((u.group == "good") & ~u.unstable).sum()),
                     has_ccf="region" in u, trials=len(tr),
                     spikes_s=f"{min(t.min() for t in trains if len(t)):.0f} to {max(t.max() for t in trains if len(t)):.0f}",
                     trials_s=f"{tr.start_time.min():.0f} to {tr.start_time.max():.0f}",
                     description=meta["description"][:60]))
files = pd.DataFrame(rows)
print(files.to_string(index=False))
paper = files[files.session.isin(estim.PAPER_SESSIONS)]
print("\nPaper recordings:", len(paper), "| probes:", sum(len(p) for p in paper.probes),
      "| probe A:", sum("A" in p for p in paper.probes), "B:", sum("B" in p for p in paper.probes),
      "C:", sum("C" in p for p in paper.probes))
print("Subjects of the paper recordings:", paper.subject.nunique(), sorted(paper.subject.unique()))
print("jlh48 and jlh49v1 (same subject id):",
      [(s, DATA[s][3]["session_id"]) for s in ["jlh48", "jlh49v1"]])

print("\nUnit groups:", pd.concat([d[0].group for d in DATA.values()]).value_counts().to_dict())
print("Unstable among good units, by recording:",
      {s: int((DATA[s][0].group.eq("good") & DATA[s][0].unstable).sum()) for s in estim.PAPER_SESSIONS})

print("\nTrials per condition (single pulses, 100 us per phase), the counts used (first 75):")
cond = {}
for s in estim.PAPER_SESSIONS:
    tr = DATA[s][1]
    cond[s] = {f"{p[0]}{a}": len(estim.condition_trials(tr, p, a, n=None)) for p in estim.POLARITIES for a in estim.AMPS}
print(pd.DataFrame(cond).T.to_string())
print("\nTrial table values: pulse_duration", sorted(pd.concat([d[1].pulse_duration for d in DATA.values()]).unique()),
      "| amplitude", sorted(pd.concat([d[1].amplitude for d in DATA.values()]).unique()),
      "| polarity", sorted(pd.concat([d[1].polarity for d in DATA.values()]).unique()))

allu = pd.concat([DATA[s][0] for s in estim.PAPER_SESSIONS])
print("\nUnit CCF ranges (um): AP %.0f to %.0f, DV %.0f to %.0f, ML %.0f to %.0f" % (
    allu.ap.min(), allu.ap.max(), allu.dv.min(), allu.dv.max(), allu.ml.min(), allu.ml.max()))
print("Distance to the first trial's cathodal contact (um): %.0f to %.0f, median %.0f" % (
    allu.distance.min(), allu.distance.max(), allu.distance.median()))
print("Signed distance (um): %.0f to %.0f" % (allu.norm_distance.min(), allu.norm_distance.max()))
print("Waveform duration (ms): %.2f to %.2f" % (allu.wf_duration.min() * 1e3, allu.wf_duration.max() * 1e3))
print("Regions of good units:", allu[allu.group == "good"].sub_cort.value_counts(dropna=False).to_dict())

In [ ]:
# The dataset's coordinates against the paper's code coordinates (outside input, a check only)
rows = []
for s in estim.PAPER_SESSIONS:
    u, tr, trains, meta = DATA[s]
    cc = CODE_COORDS[s]
    off = []
    for c in tr.contact_negative.unique():
        ds = tr.loc[tr.contact_negative == c, ["neg_ap", "neg_dv", "neg_ml"]].iloc[0].to_numpy()
        off.append(np.round(ds - cc["stim"][estim.NN_CONTACTS.index(int(c))]).astype(int).tolist())
    dd = []
    for i, r in u.iterrows():
        arr = cc.get("probe" + r.probe)
        if arr is None:
            continue
        dd.append(np.linalg.norm(arr[min(r.ch, len(arr) - 1)] - np.array([r.ap, r.dv, r.ml])))
    dd = np.array(dd)
    k = u.keep.to_numpy()
    rows.append(dict(session=s, contact_offset_um=off[0], contacts_same_offset=all(o == off[0] for o in off),
                     unit_offset_median_um=round(float(np.median(dd)), 1),
                     units_over_100um=int((dd > 100).sum()),
                     distance_diff_median_um=round(float(np.nanmedian(np.abs(u.distance - u.distance_code)[k])), 1)))
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Clock check: population spikes in 0 to 20 ms after 100 uA pulses (cathodal, or bipolar where
# there is none) against the 20 ms before, with the true onsets and with onsets moved 1 s later
# (half way between pulses, which are 2 s apart). Paired Wilcoxon over trials.
rows = []
for s in estim.PAPER_SESSIONS:
    u, tr, trains, meta = DATA[s]
    pol = "cathodal" if len(estim.condition_trials(tr, "cathodal", 100)) else "bipolar"
    pop = np.sort(np.concatenate([trains[i] for i in np.where(u.keep)[0]]))
    res = {"session": s, "polarity": pol}
    for label, shift in [("true", 0.0), ("shifted", 1.0)]:
        st = estim.condition_trials(tr, pol, 100).start_time.to_numpy() + shift
        post = np.searchsorted(pop, st + 0.02) - np.searchsorted(pop, st)
        pre = np.searchsorted(pop, st) - np.searchsorted(pop, st - 0.02)
        res[f"{label}_post_over_pre"] = round(post.mean() / pre.mean(), 2)
        res[f"{label}_p"] = stats.wilcoxon(post, pre).pvalue if (post != pre).any() else 1.0
    rows.append(res)
print(pd.DataFrame(rows).to_string(index=False))

The checks against the description:

- Ten files; nine are the recordings the paper's code loads, and the tenth (jlh41, "lots of noise
  issues ... not much analyzed yet here") has no CCF coordinates or regions; it is left out. The nine
  give the paper's 25 probes (A 7, B 9, C 9). They come from eight subject ids: the file of session
  jlh49v1 is filed under subject jlh48, which the paper and its code treat as a separate recording
  and mouse; its age (P381D) is one day older than the jlh48 file recorded four months later.
- The electrodes table holds placeholders (location "none", constant coordinates), so channel CCF
  coordinates come from the units' own coordinates.
- Conditions are labelled "monopolar" or "bipolar" with signed amplitudes; cathodal and anodal are
  the sign. Cathodal single pulses are in seven of the nine recordings (jlh39 and jlh49v1 hold only
  bipolar pulses, jlh40 no bipolar), so the claims on cathodal pulses rest on seven recordings, where
  the paper's legends say nine. jlh33 has 60 trials per condition, where the Methods say 75; its
  anodal 50 uA single pulse is missing.
- Trial onsets and spike times share a clock: population firing in the 20 ms after a 100 uA pulse
  differs from the 20 ms before it with the true onsets, and not with onsets moved 1 s later
  (table above). Two files (jlh40,
  jlh53concat) have negative times; this does not affect the alignment.
- Unit coordinates agree with the paper's code coordinates at the unit's channel to a median of
  about 24 um; units beyond the end of the code's channel arrays differ more. The stimulation
  contact coordinates of jlh31 and jlh32 differ from the code's by 206 and 373 um; the other seven
  agree exactly. Distances are computed from the dataset's coordinates, with the code's shown as a
  variant where they matter.

In [ ]:
# Direct response table: nDRP per good stable unit and condition (Methods, DRP)
def drp_table(sessions=estim.PAPER_SESSIONS, win=0.002, base=(0.007, 0.005), polarities=estim.POLARITIES,
              contact_choice=None):
    rows = []
    for s in sessions:
        u, tr, trains, meta = DATA[s]
        idx = np.where(u.keep)[0]
        for pol in polarities:
            for a in estim.AMPS:
                contact = contact_choice.get((s, pol)) if contact_choice else None
                c = estim.condition_trials(tr, pol, a, contact=contact)
                if len(c) == 0:
                    continue
                st = c.start_time.to_numpy()
                for i in idx:
                    p, pb, n = estim.spike_prob(trains[i], st, win=win, base=base)
                    if np.isnan(p):
                        continue
                    rows.append((s, i, pol, a, p, pb, n))
    df = pd.DataFrame(rows, columns=["session", "unit", "pol", "amp", "drp", "base", "ndrp"])
    meta_cols = ["distance", "norm_distance", "distance_code", "norm_distance_code", "probe", "region",
                 "sub_cort", "wf_kmeans", "wf_threshold"]
    parts = []
    for s, g in df.groupby("session"):
        parts.append(g.join(DATA[s][0][meta_cols], on="unit"))
    return pd.concat(parts, ignore_index=True)

DRP = drp_table()
CAT = DRP[DRP.pol == "cathodal"]
RESP = CAT[CAT.ndrp > 0.15]
print("Unit and condition values:", len(DRP), "| cathodal:", len(CAT))
print(CAT.groupby("amp").agg(units=("ndrp", "size"), responders=("ndrp", lambda x: int((x > 0.15).sum())),
                             proportion=("ndrp", lambda x: (x > 0.15).mean())).round(4).to_string())
print("Responders by recording and amplitude:")
print(RESP.groupby(["session", "amp"]).size().unstack(fill_value=0).to_string())
print("Unit values per condition:", DRP.groupby(["pol", "amp"]).size().to_dict())

## fig4d-latency-trough

> First-spike latencies after a pulse are bimodal, with an early peak, a pause at a median of 1.95 ms
> across parameters and a broader peak from about 2.5 ms, which sets the 2 ms direct window.

First spike after each pulse (within 500 ms) for all good units, all trials of the condition's
contact (the code's choose_stim_parameter without the 75-trial cut, figure4_figure5_figure6.ipynb
cell 20), 0.1 ms bins over 0 to 15 ms, trough the lowest bin with centre in 0.5 to 5 ms (cell 22).

In [ ]:
def latencies(pol, a, n=None, stable=False):
    out = []
    for s in estim.PAPER_SESSIONS:
        u, tr, trains, meta = DATA[s]
        c = estim.condition_trials(tr, pol, a, n=n)
        if len(c) == 0:
            continue
        st = c.start_time.to_numpy()
        sel = u.keep if stable else (u.group == "good")
        for i in np.where(sel)[0]:
            out.append(estim.first_latencies(trains[i], st))
    return np.concatenate(out)

EDGES = np.arange(0, 15.1, 0.1)
def trough(lat):
    cnt, e = np.histogram(lat, EDGES)
    ctr = e[:-1] + 0.05
    m = (ctr >= 0.5) & (ctr <= 5)
    j = np.where(m)[0][np.argmin(cnt[m])]
    return ctr[j], cnt[j], cnt[:j].max(), cnt[j + 1:].max(), ctr[j + 1:][np.argmax(cnt[j + 1:])]

LAT = {}
rows = []
for pol in estim.POLARITIES:
    for a in estim.AMPS:
        lat = latencies(pol, a)
        LAT[(pol, a)] = lat
        t, c, before, after, after_at = trough(lat)
        t75 = trough(latencies(pol, a, n=75, stable=True))[0]
        rows.append(dict(pol=pol, amp=a, n_latencies=int(((lat >= 0) & (lat <= 15)).sum()), trough_ms=round(t, 2),
                         trough_count=c, max_before=before, max_after=after, after_peak_ms=round(after_at, 2),
                         trough_first75_stable=round(t75, 2)))
lt = pd.DataFrame(rows)
print(lt.to_string(index=False))
print("\nMedian trough, cathodal (criterion):", lt[lt.pol == "cathodal"].trough_ms.median(),
      "| all 12 conditions:", lt.trough_ms.median(),
      "| first 75 trials, stable units, cathodal:", lt[lt.pol == "cathodal"].trough_first75_stable.median())
print("Conditions with a higher bin before and after the trough:", int(((lt.max_before > lt.trough_count) & (lt.max_after > lt.trough_count)).sum()), "of", len(lt))
fig, ax = plt.subplots(1, 4, figsize=(11, 2.5), sharey=False)
for k, a in enumerate(estim.AMPS):
    ax[k].hist(LAT[("cathodal", a)], EDGES, color="k")
    ax[k].axvline(lt[(lt.pol == "cathodal") & (lt.amp == a)].trough_ms.iloc[0], color="r")
    ax[k].set_title(f"cathodal {a} uA"); ax[k].set_xlabel("first spike latency (ms)")
plt.tight_layout(); plt.show()
print("All latencies plotted lie in 0 to 15 ms by the bins; latencies beyond 15 ms are not shown.")

**Result: reproduced.** The trough of the first-spike latency histogram lies at 1.95, 1.75, 1.95
and 1.95 ms for 5 to 100 uA cathodal pulses, a median of 1.95 ms, with a higher bin before and after
it in all 12 conditions. Across all 12 conditions the median is 1.45 ms (anodal and bipolar troughs
at 1.05 to 1.45 ms); the criterion takes the cathodal amplitudes. The later cathodal peaks lie at
3.75 to 11.85 ms.

## figs3cd-gmm-window

> Gaussian mixtures of first-spike latencies give amplitude-specific direct windows (0.68, 1.14, 1.22
> and 1.47 ms at 5, 25, 50 and 100 uA cathodal), and the proportion of direct responders does not
> differ from that of the 2 ms window.

Latencies of fig4d (0 to 15 ms), one-dimensional GMM (sklearn, maximum likelihood, 10
initialisations, seeded), readings (a) 4 components and (b) the number of components (1 to 4) with
the lowest BIC. The window is where the posterior of the second-earliest component first equals that
of the earliest (Methods, Gaussian mixture models). Responders per recording with each window and
the 2 ms window, Wilcoxon signed-rank over recordings, FDR (Benjamini-Hochberg) over amplitudes.

In [ ]:
rng_gmm = np.random.default_rng(3)
GRID = np.linspace(0, 15, 15001)
def gmm_window(x, k, seed):
    g = GaussianMixture(k, n_init=10, random_state=seed).fit(x[:, None])
    order = np.argsort(g.means_.ravel())
    if k < 2:
        return np.nan, g
    post = g.predict_proba(GRID[:, None])[:, order]
    m0 = g.means_.ravel()[order][0]
    after = GRID > m0
    cross = np.where(after & (post[:, 1] >= post[:, 0]))[0]
    return (GRID[cross[0]] if len(cross) else np.nan), g

WIN = {}
rows = []
for a in estim.AMPS:
    x = LAT[("cathodal", a)]
    x = x[(x >= 0) & (x <= 15)]
    seed = int(rng_gmm.integers(1e9))
    w4, g4 = gmm_window(x, 4, seed)
    bics = {k: GaussianMixture(k, n_init=10, random_state=seed).fit(x[:, None]).bic(x[:, None]) for k in range(1, 5)}
    kb = min(bics, key=bics.get)
    wb, _ = gmm_window(x, kb, seed)
    WIN[a] = {"a": w4, "b": wb}
    rows.append(dict(amp=a, n=len(x), window_k4_ms=round(w4, 3), means_k4=np.round(np.sort(g4.means_.ravel()), 2).tolist(),
                     bic_k=kb, window_bic_ms=round(wb, 3) if np.isfinite(wb) else np.nan))
print(pd.DataFrame(rows).to_string(index=False))
print("Expected 0.68, 1.14, 1.22, 1.47 ms (tolerance 10%)")

CAT_SESS = sorted(CAT.session.unique())
def prop_responders(win, base):
    out = {}
    for s in CAT_SESS:
        u, tr, trains, meta = DATA[s]
        idx = np.where(u.keep)[0]
        out[s] = {}
        for a in estim.AMPS:
            st = estim.condition_trials(tr, "cathodal", a).start_time.to_numpy()
            w = win[a] if isinstance(win, dict) else win
            v = np.array([estim.spike_prob(trains[i], st, win=w / 1000, base=base(w))[2] for i in idx])
            v = v[np.isfinite(v)]
            out[s][a] = np.mean(v > 0.15)
    return pd.DataFrame(out).T

P2 = prop_responders(2.0, lambda w: (0.007, 0.005))
for reading in ["a", "b"]:
    win = {a: WIN[a][reading] for a in estim.AMPS}
    if not all(np.isfinite(list(win.values()))):
        print(f"reading {reading}: no window at some amplitude, {win}")
        continue
    for bname, base in [("(i) baseline as long as the window", lambda w: (0.005 + w / 1000, 0.005)),
                        ("(ii) 2 ms baseline", lambda w: (0.007, 0.005))]:
        PG = prop_responders(win, base)
        p = [stats.wilcoxon(PG[a], P2[a]).pvalue if (PG[a] != P2[a]).any() else 1.0 for a in estim.AMPS]
        padj = multipletests(p, method="fdr_bh")[1]
        allp = stats.wilcoxon(PG.to_numpy().ravel(), P2.to_numpy().ravel()).pvalue
        print(f"\nGMM reading {reading}, {bname}: mean proportion GMM {PG.mean().round(4).to_dict()} | 2 ms {P2.mean().round(4).to_dict()}")
        print("   Wilcoxon p", np.round(p, 4).tolist(), "FDR", np.round(padj, 4).tolist(), "| pooled over amplitudes p", round(allp, 4))
print("\nRecordings:", len(CAT_SESS))

**Result: reproduced.** Four-component mixtures put the direct window at 0.683, 1.139, 1.221 and
1.470 ms for 5 to 100 uA, and BIC also chooses four components at every amplitude. Over the seven
recordings with cathodal pulses, the proportion of responders with these windows does not differ from
the 2 ms window at any amplitude (smallest FDR-adjusted p 0.25, under both baselines). Pooled over the
four amplitudes, the GMM windows give fewer responders (Wilcoxon p = 0.0022), a test the legend does
not name.

## fig4h-extent-constant

> The spatial extent of direct responders stays relatively constant across amplitudes (1180 +- 559,
> 884 +- 652, 965 +- 451 and 522 +- 313 um), with no significant amplitude effect.

Per recording and cathodal amplitude, extent = |minimum| + maximum of the responders' signed
distances (code cell 30); one-way ANOVA across amplitudes of the per-recording extents.

In [ ]:
for col in ["norm_distance", "norm_distance_code"]:
    ext = RESP.groupby(["amp", "session"])[col].agg(lambda d: abs(d.min()) + d.max()).reset_index()
    groups = [g[col].to_numpy() for _, g in ext.groupby("amp")]
    F, p = stats.f_oneway(*groups)
    print(f"{col}: mean +- SD per amplitude", {a: f"{g[col].mean():.0f} +- {g[col].std(ddof=0):.0f} (n={len(g)})" for a, g in ext.groupby("amp")},
          f"| one-way ANOVA F = {F:.3f}, p = {p:.3f}")
    if col == "norm_distance":
        print(ext.pivot(index="session", columns="amp", values=col).round(0).to_string())
print("Expected 1180, 884, 965, 522 um (10%); paper F = 1.265, p = 0.312")

**Result: reproduced.** The per-recording extent of responders does not change with amplitude
(one-way ANOVA F = 1.250, p = 0.317), with means of 1121, 825, 813 and 452 um; the last two are
outside 10% of the paper's 965 and 522 um. With the paper's code coordinates the means are 1179, 884,
868 and 522 um and F = 1.265, p = 0.312, the paper's values.

## fig4h-distance-decreases

> Higher amplitudes bring direct responders closer to the stimulation site (mean distance 426, 300,
> 137 and 146 um at 5, 25, 50 and 100 uA), driven by 5 against 50 and 5 against 100 uA.

In [ ]:
for col in ["norm_distance", "norm_distance_code"]:
    g = RESP.groupby("amp")[col]
    F, p = stats.f_oneway(*[x.to_numpy() for _, x in g])
    print(f"{col}: mean (SD, n)", {a: f"{x.mean():.0f} ({x.std():.0f}, {len(x)})" for a, x in g}, f"| ANOVA F = {F:.2f}, p = {p:.4f}")
    tk = pairwise_tukeyhsd(RESP[col], RESP.amp)
    print(pd.DataFrame(tk.summary().data[1:], columns=tk.summary().data[0]).to_string(index=False))
    print("By recording (mean, n):", RESP.groupby(["session", "amp"])[col].agg(lambda x: f"{x.mean():.0f} ({len(x)})").unstack().to_string())
print("Expected 426, 300, 137, 146 um (10%); paper F = 4.15, p = 0.007; Tukey 5-50 p = 0.015, 5-100 p = 0.019")
print("Range of signed distances of responders: %.0f to %.0f um" % (RESP.norm_distance.min(), RESP.norm_distance.max()))

**Result: partially reproduced.** Mean signed distances are 414, 264, 129 and 142 um (ANOVA
F = 3.97, p = 0.0094; Tukey 5 against 50 uA p = 0.0134, 5 against 100 uA p = 0.0199, no other pair
below 0.05). The tests hold; the 25 uA mean (264 um) is outside 10% of 300 um. With the paper's code
coordinates the means are 426, 300, 137 and 146 um and F = 4.15, p = 0.0074, the paper's values: the
difference is the stimulation contact coordinates of jlh31 and jlh32 in the dataset.

## fig4h-ks-distinct

> 5 and 100 uA recruit distinct spatial populations of direct responders; the other pairs the paper
> reports (5-25, 25-50, 25-100 and 50-100 uA) do not differ after correction.

In [ ]:
for col in ["norm_distance", "norm_distance_code"]:
    pairs = [(5, 25), (5, 50), (5, 100), (25, 50), (25, 100), (50, 100)]
    p = [stats.ks_2samp(RESP[RESP.amp == x][col], RESP[RESP.amp == y][col]).pvalue for x, y in pairs]
    adj = estim.holm_sidak(p)
    D = [stats.ks_2samp(RESP[RESP.amp == x][col], RESP[RESP.amp == y][col]).statistic for x, y in pairs]
    print(col)
    print(pd.DataFrame(dict(pair=[f"{x}-{y}" for x, y in pairs], D=np.round(D, 3), p=np.round(p, 4), p_holm_sidak=np.round(adj, 4))).to_string(index=False))
print("Paper adjusted p: 5-25 0.566, 5-100 0.016, 25-50 0.566, 25-100 0.363, 50-100 0.083")

**Result: not reproduced.** With the dataset's coordinates, 5 against 100 uA has KS p = 0.0193 and
Holm-Sidak adjusted p = 0.1105, so the 5 and 100 uA populations do not differ after correction; the
other four pairs do not differ (adjusted p 0.5255 to 0.8290). With the paper's code coordinates the
adjusted p are 0.5662 (5-25), 0.0164 (5-100), 0.5662 (25-50), 0.3625 (25-100) and 0.8245 (50-100),
which reproduce the claim (the paper prints 0.083 for 50-100).

## fig4j-drp-increases

> Among units responsive at any amplitude (99 units), direct response probability increases with
> amplitude (0.12, 0.14, 0.19 and 0.23).

In [ ]:
PIV = CAT.pivot_table(index=["session", "unit"], columns="amp", values="ndrp")
resp_ids = RESP.set_index(["session", "unit"]).index.unique()
R99 = PIV.loc[resp_ids].dropna()
print("Responders at any cathodal amplitude:", len(resp_ids), "| with a value at all four:", len(R99))
print("Mean +- SD nDRP:", {a: f"{R99[a].mean():.3f} +- {R99[a].std():.3f}" for a in estim.AMPS})
long = R99.stack().rename("ndrp").reset_index()
m = smf.ols("ndrp ~ amp", data=long).fit()
print("(a) OLS on numeric amplitude:", anova_lm(m, typ=2).loc["amp", ["F", "PR(>F)"]].round(5).to_dict(), "slope", round(m.params["amp"], 5))
print("(b) one-way ANOVA, amplitude categorical: F = %.2f, p = %.2e" % stats.f_oneway(*[R99[a] for a in estim.AMPS]))
rm = AnovaRM(long.assign(uid=long.session + "_" + long.unit.astype(str)), "ndrp", "uid", within=["amp"]).fit()
print("(c) repeated-measures ANOVA:", rm.anova_table.round(5).to_dict("records"))
print("By recording, responders:", R99.groupby(level=0).size().to_dict())
print("Expected 99 units (10%), 0.12, 0.14, 0.19, 0.23; paper F = 12.78, p = 0.0004")

**Result: reproduced.** 99 units respond at some cathodal amplitude and have values at all four;
their mean nDRP is 0.116, 0.142, 0.194 and 0.233, and the code's regression on numeric amplitude gives
F = 12.779, p = 0.00039 (paper F = 12.78, p = 0.0004). A one-way ANOVA with amplitude categorical
(F = 4.44, p = 0.0044) and a repeated-measures ANOVA (F = 4.66, p = 0.0034) agree.

## figs3e-early-windows

> Broader windows (5 and 10 ms) give more responsive units than the 2 ms window (0.092 and 0.175 at
> 100 uA), and their amplitude dependence is stronger.

Proportion of good stable units with nDRP > 0.15 per recording, windows of 2, 5 and 10 ms; baseline
readings (a) 2 ms ending 3 ms plus the window before onset, (b) as long as the window, ending 3 ms
before onset; the 2 ms comparator is the main DRP. Paired Wilcoxon over recordings, FDR over the
eight comparisons.

In [ ]:
for reading, base in [("a", lambda w: (0.005 + w / 1000, 0.003 + w / 1000)), ("b", lambda w: (0.003 + w / 1000, 0.003))]:
    PW = {w: prop_responders(float(w), base) for w in (5, 10)}
    print(f"Reading {reading}: mean (SEM) proportion over {len(P2)} recordings")
    tab = pd.DataFrame({f"{w} ms": [f"{d[a].mean():.3f} ({d[a].sem():.3f})" for a in estim.AMPS] for w, d in [(2, P2)] + list(PW.items())}, index=estim.AMPS)
    print(tab.to_string())
    p, lab = [], []
    for w in (5, 10):
        for a in estim.AMPS:
            p.append(stats.wilcoxon(PW[w][a], P2[a]).pvalue if (PW[w][a] != P2[a]).any() else 1.0)
            lab.append(f"{w} ms vs 2 ms, {a} uA")
    adj = multipletests(p, method="fdr_bh")[1]
    print(pd.DataFrame(dict(comparison=lab, p=np.round(p, 4), p_fdr=np.round(adj, 4))).to_string(index=False))
    print("Increase 5 to 100 uA in mean proportion: 2 ms %.3f, 5 ms %.3f, 10 ms %.3f\n" % (
        P2[100].mean() - P2[5].mean(), PW[5][100].mean() - PW[5][5].mean(), PW[10][100].mean() - PW[10][5].mean()))
print("Expected at 100 uA: 5 ms 0.092, 10 ms 0.175 (10%)")

**Result: partially reproduced.** Broader windows give more responders and a stronger amplitude
dependence: from 5 to 100 uA the mean proportion rises by 0.005 with 2 ms, 0.041 with 5 ms and 0.073
with 10 ms (reading a), and at 100 uA both broader windows exceed 2 ms (FDR p = 0.0208 under reading
a, 0.0417 under b). The 100 uA proportions are 0.058 and 0.106 (reading a) or 0.057 and 0.091 (b),
below 10% of the paper's 0.092 and 0.175. With seven recordings the smallest exact p is 0.0156.

## fig5bc-amplitude-clusters

> Clustering the direct responders by response probability across amplitude gives six clusters,
> broadly dose-dependent (58.6% of responders) or amplitude-selective, responding almost only at one
> amplitude (19.2% at 5 uA, 15.2% at 25 uA, 7.1% at 100 uA).

The 99 responders of fig4j; Ward linkage, maxclust 6; features (a) the four nDRP values, as the code,
(b) each unit's vector divided by its maximum. A cluster is selective when its mean peaks at one
amplitude and is below half the peak at every other (the claims file's rule). The fractions are also
checked over every assignment of clusters to the four labels (the criterion's "some assignment"), and
over those in which a cluster labelled selective for an amplitude peaks at that amplitude.

In [ ]:
import itertools
EXP_AMP = {"dose": 0.586, 5: 0.192, 25: 0.152, 100: 0.071}

def cluster_summary(X, k=6):
    Z = linkage(X, method="ward")
    lab = fcluster(Z, k, criterion="maxclust")
    ccc = cophenet(Z, pdist(X))[0]
    sil = silhouette_score(X, lab)
    return Z, lab, ccc, sil

for reading, X in [("a", R99.to_numpy()), ("b", (R99.T / R99.max(axis=1)).T.to_numpy())]:
    Z, lab, ccc, sil = cluster_summary(X)
    rows = []
    for c in np.unique(lab):
        m = R99.to_numpy()[lab == c].mean(0)
        pk = estim.AMPS[int(np.argmax(m))]
        sel = all(m[j] < 0.5 * m.max() for j in range(4) if estim.AMPS[j] != pk)
        rows.append(dict(cluster=c, n=int((lab == c).sum()), frac=round((lab == c).mean(), 3),
                         mean_ndrp=np.round(m, 3).tolist(), peak_uA=pk, selective=sel))
    cl = pd.DataFrame(rows)
    fr = {"dose": cl[~cl.selective].frac.sum()}
    for a in estim.AMPS:
        fr[a] = cl[cl.selective & (cl.peak_uA == a)].frac.sum()
    print(f"Reading {reading}: CCC = {ccc:.3f}, silhouette = {sil:.3f}")
    print(cl.to_string(index=False))
    print("   fractions by the half-peak rule:", {k: round(v, 3) for k, v in fr.items()})
    ok = []
    for assign in itertools.product(["dose", 5, 25, 100], repeat=len(cl)):
        if any(l != "dose" and l != p for l, p in zip(assign, cl.peak_uA)):
            continue
        f = {key: sum(cl.frac[i] for i, l in enumerate(assign) if l == key) for key in EXP_AMP}
        if all(abs(f[key] - EXP_AMP[key]) <= 0.1 * EXP_AMP[key] for key in EXP_AMP):
            ok.append(assign)
    print("   assignments with selective labels at the cluster's peak, matching all four fractions within 10%:", len(ok), ok[:3])
    ok_any = []
    for assign in itertools.product(["dose", 5, 25, 100], repeat=len(cl)):
        f = {key: sum(cl.frac[i] for i, l in enumerate(assign) if l == key) for key in EXP_AMP}
        if all(abs(f[key] - EXP_AMP[key]) <= 0.1 * EXP_AMP[key] for key in EXP_AMP):
            ok_any.append(assign)
    print("   any assignment of clusters to the labels matching within 10%:", len(ok_any), ok_any[:3])
print("Expected CCC 0.702, silhouette 0.352 (10%)")

**Result: reproduced.** On the code's features (reading a) the 99 responders give CCC = 0.702 and
silhouette = 0.352, the paper's values, and clusters of 15, 19, 40, 13, 7 and 5 units. Labelling the
clusters 25 uA, 5 uA, dose, dose, 100 uA and dose gives the paper's 15.2%, 19.2%, 58.6% and 7.1%, the
assignment the criterion allows. By the half-peak rule the cluster of 7 peaks at 50 uA (mean nDRP 0.651
against 0.343 at 100 uA) and the cluster of 13 is the one selective for 100 uA, so no assignment that
follows the clusters' peaks matches; reading (b) gives CCC 0.840 and silhouette 0.501 and no matching
assignment.

## fig5ef-polarity-clusters

> Clustering responders by their response to cathodal, anodal and bipolar pulses shows a polarity
> bias: 69.1% are polarity-specific, preferring bipolar (27.7%), anodal (23.6%) or cathodal (17.7%)
> pulses.

As the code (cells 77 to 81): per polarity the maximum nDRP over amplitudes, missing polarities 0,
divided by the unit's maximum; Ward, maxclust 6. Populations (a) units with nDRP > 0.15 in any
polarity and amplitude, (b) the 99 cathodal responders. A cluster prefers a polarity when its mean is
highest there and below half of that for both others. Variant: for bipolar pulses, the contact pair
whose cathodal contact lies nearest layer 5 units (the Methods' "in or near Layer 5"), in place of
the first pair.

In [ ]:
EXP_POL = {"bipolar": 0.277, "anodal": 0.236, "cathodal": 0.177}

def polarity_clusters(drp, ids):
    sub = drp.set_index(["session", "unit"]).loc[ids].reset_index()
    feat = sub.groupby(["session", "unit", "pol"]).ndrp.max().unstack(fill_value=0.0)
    feat = feat.reindex(columns=["bipolar", "cathodal", "anodal"], fill_value=0.0)
    feat = feat.div(feat.max(axis=1), axis=0).fillna(0.0)
    X = feat.to_numpy()
    Z, lab, ccc, sil = cluster_summary(X)
    rows = []
    for c in np.unique(lab):
        m = X[lab == c].mean(0)
        top = int(np.argmax(m))
        pref = feat.columns[top] if all(m[j] < 0.5 * m[top] for j in range(3) if j != top) else "none"
        rows.append(dict(cluster=c, n=int((lab == c).sum()), frac=round((lab == c).mean(), 3),
                         mean=np.round(m, 2).tolist(), prefers=pref))
    cl = pd.DataFrame(rows)
    fr = {p: round(cl[cl.prefers == p].frac.sum(), 3) for p in EXP_POL}
    return len(X), ccc, sil, cl, fr

all_ids = DRP[DRP.ndrp > 0.15].set_index(["session", "unit"]).index.unique()
# layer 5 variant: per recording, the bipolar cathodal contact nearest layer 5 units
L5 = {}
for s in estim.PAPER_SESSIONS:
    u, tr = DATA[s][0], DATA[s][1]
    regs = estim.contact_regions(u, tr)
    bip = tr[(tr.pol == "bipolar") & (tr.pulse_duration == 100) & (tr.pulse_number == 1)]
    cands = [int(c) for c in bip.contact_negative.unique()]
    l5 = [c for c in cands if "layer 5" in regs[c][0]]
    if l5:
        L5[(s, "bipolar")] = l5[0]
print("Bipolar contacts (first used by the code, layer 5 variant):",
      {s: (int(DATA[s][1][DATA[s][1].pol == 'bipolar'].contact_negative.iloc[0]) if (DATA[s][1].pol == 'bipolar').any() else None, L5.get((s, 'bipolar'))) for s in estim.PAPER_SESSIONS})
DRP_L5 = pd.concat([DRP[DRP.pol != "bipolar"], drp_table(polarities=["bipolar"], contact_choice=L5)], ignore_index=True)
for name, drp, ids in [("(a) any polarity", DRP, all_ids), ("(b) 99 cathodal responders", DRP, R99.index),
                       ("(a) any polarity, layer 5 bipolar contact", DRP_L5, DRP_L5[DRP_L5.ndrp > 0.15].set_index(["session", "unit"]).index.unique())]:
    n, ccc, sil, cl, fr = polarity_clusters(drp, ids)
    print(f"\n{name}: n = {n}, CCC = {ccc:.3f}, silhouette = {sil:.3f}, polarity-specific = {sum(fr.values()):.3f}, {fr}")
    print(cl.to_string(index=False))
print("\nExpected CCC 0.806, silhouette 0.519, polarity-specific 0.691, bipolar 0.277, anodal 0.236, cathodal 0.177 (10%)")

**Result: partially reproduced.** Under reading (a), the code's population, 220 units give
CCC = 0.773 and silhouette = 0.538 (within 10% of 0.806 and 0.519) and three clusters that prefer one
polarity: anodal 0.232 and cathodal 0.186 match, bipolar 0.205 is below 0.277, and the
polarity-specific total is 0.623 (paper 0.691). With the bipolar contact nearest layer 5 (235 units)
the fractions are 0.260, 0.213 and 0.196 (cathodal just outside 0.177 +- 0.018). Among the 99 cathodal
responders (reading b) no cluster prefers bipolar or anodal pulses.

## fig5-fsrs-proportion

> Direct responders stay sparse for both FS and RS units across amplitudes, with a main effect of
> amplitude but no effect of waveform class or interaction.

Responder indicator (nDRP > 0.15) over FS or RS good stable units pooled across recordings, cathodal;
two-way ANOVA (type II) of the indicator on amplitude and class. Classes under the Methods' k-means
and the code's 0.4 ms threshold. Reading (b) of the criterion, units within the evoked potential,
needs the evoked-potential boundaries, which the dandiset cannot give, and is not run.

In [ ]:
for cls_col in ["wf_kmeans", "wf_threshold"]:
    d = CAT[CAT[cls_col].isin(["fs", "rs"])].assign(resp=lambda x: (x.ndrp > 0.15).astype(float), cls=lambda x: x[cls_col])
    print(f"Classes by {cls_col}: units per class", d.groupby("cls").unit.size().div(4).round().to_dict())
    print(d.groupby(["cls", "amp"]).resp.agg(["mean", "std", "size"]).round(4).unstack(0).to_string())
    for amp_term in ["C(amp)", "amp"]:
        t = anova_lm(smf.ols(f"resp ~ {amp_term} * C(cls)", data=d).fit(), typ=2)
        print(f"   amplitude {'categorical' if 'C' in amp_term else 'numeric'}:",
              {k: f"F = {t.loc[k, 'F']:.2f}, p = {t.loc[k, 'PR(>F)']:.3f}" for k in t.index[:-1]})
print("Expected FS 0.032 (5 uA), 0.040 (100 uA); RS 0.013 (5 uA), 0.037 (100 uA) (10%); paper amplitude F = 3.04, p = 0.028; class p = 0.262; interaction p = 0.210")

**Result: partially reproduced, resting on an assumption.** Responders are sparse in both classes
(1.0% to 2.5% of units under the code's classifier) and amplitude has an effect when it is numeric (F = 4.33,
p = 0.037) but not when categorical (p = 0.123). Under the code's classifier, class (p = 0.437) and
interaction (p = 0.150) have no effect; under the Methods' k-means, class has one (p = 0.028). The
proportions (FS 0.0196 and 0.0227, RS 0.0111 and 0.0221 at 5 and 100 uA) are outside 10% of the
paper's 0.032, 0.040, 0.013 and 0.037; the paper's values may be those of units within the evoked
potential (reading b), which the dandiset cannot give.

## fig5j-fs-closer

> FS direct responders lie closer to the stimulation site than RS responders, and higher amplitude
> brings both closer.

As the code (cell 57): signed distance of cathodal responders classed FS or RS,
`ols('value ~ C(amplitude) * C(cell_type)')`, type II.

In [ ]:
for cls_col in ["wf_kmeans", "wf_threshold"]:
    for col in ["norm_distance", "norm_distance_code"]:
        d = RESP[RESP[cls_col].isin(["fs", "rs"])].assign(cls=lambda x: x[cls_col], value=lambda x: x[col])
        t = anova_lm(smf.ols("value ~ C(amp) * C(cls)", data=d).fit(), typ=2)
        means = d.groupby(["cls", "amp"]).value.agg(lambda x: f"{x.mean():.0f} (n={len(x)})").unstack()
        print(f"{cls_col}, {col}:", {k: f"F = {t.loc[k, 'F']:.2f}, p = {t.loc[k, 'PR(>F)']:.4f}" for k in t.index[:-1]},
              "| mean FS %.0f, RS %.0f" % (d[d.cls == "fs"].value.mean(), d[d.cls == "rs"].value.mean()))
        print(means.to_string())
print("Expected FS 205 (5 uA), RS 567 (5 uA), RS 185 (100 uA) um (10%); paper waveform F = 8.04, p = 0.005; amplitude F = 4.10, p = 0.008")

**Result: reproduced, resting on an assumption.** With the classes of the paper's code (duration
threshold), FS responders lie closer than RS responders (mean 111 against 267 um; waveform F = 7.02,
p = 0.0090) and amplitude has an effect (F = 3.95, p = 0.0097); FS 205 um and RS 556 and 183 um at 5
and 100 uA are within 10% of 205, 567 and 185 um. With the code's coordinates the paper's F = 8.04
and 4.10 are matched exactly. Under the Methods' k-means classes the waveform effect is not significant
(p = 0.0874).

## figs6b-fsrs-early-firing

> In the 10 ms after a pulse, the firing of FS and RS cortical units differs at every amplitude.

From the S6B legend and the code (cells 62 to 67): per recording, cortical good stable units of each
class, first 75 cathodal trials; spikes of the class pooled, counted per trial in bins, averaged over
trials and divided by the number of units; the bins from 0 to 10 ms of every recording pooled per
class; two-sample KS test per amplitude, Holm-Sidak over the four. Bin widths 1 ms (code) and 2 ms
(legend).

In [ ]:
for cls_col in ["wf_kmeans", "wf_threshold"]:
    for bw in [0.001, 0.002]:
        edges = np.arange(0, 0.010 + bw / 2, bw)
        p, ns, med = [], [], []
        for a in estim.AMPS:
            samp = {"fs": [], "rs": []}
            for s in CAT_SESS:
                u, tr, trains, meta = DATA[s]
                st = estim.condition_trials(tr, "cathodal", a).start_time.to_numpy()
                for c in ["fs", "rs"]:
                    idx = np.where(u.keep & (u.sub_cort == "Cortical") & (u[cls_col] == c))[0]
                    if len(idx) == 0:
                        continue
                    pop = np.sort(np.concatenate([trains[i] for i in idx]))
                    samp[c].extend(estim.counts_matrix(pop, st, edges).mean(0) / len(idx))
            p.append(stats.ks_2samp(samp["fs"], samp["rs"]).pvalue)
            ns.append((len(samp["fs"]), len(samp["rs"])))
            med.append((round(np.mean(samp["fs"]), 4), round(np.mean(samp["rs"]), 4)))
        adj = estim.holm_sidak(p)
        print(f"{cls_col}, {bw * 1000:.0f} ms bins: p", np.round(p, 4).tolist(), "| Holm-Sidak", np.round(adj, 4).tolist(),
              "| bins (FS, RS)", ns, "| mean spikes per unit and bin (FS, RS)", med)
print("Paper adjusted p: 0.010, 0.007, 0.019, 0.010")

**Result: reproduced, resting on an assumption.** With the code's classes and 1 ms bins, FS and RS
early firing differ at every amplitude (Holm-Sidak p 0.0056, 0.0038, 0.0127, 0.0127). With 2 ms bins
(legend) no adjusted p is below 0.05 (0.0598 to 0.1151), and under the Methods' k-means classes only
100 uA differs (1 ms bins, adjusted p 0.0139).

## fig6c-modulation-cortex

> Spiking modulation after a single pulse increases with amplitude for cortical units (12.60, 13.37,
> 15.11, 20.21) but stays about constant for subcortical units (12.27, 12.60, 12.08, 13.48),
> differing between them at 50 and 100 uA.

MI per bin against the median rate 1 s to 0.5 s before onset; magnitude = trapezoid area of |MI|
over bins (code, figure7.ipynb cells 28 and 33). Readings (a) 0 to 500 ms, 10 ms bins (code);
(b) 0 to 500 ms, 5 ms bins (legend); (c) 500 to 1000 ms, 5 ms bins (Methods). Good stable units,
first 75 cathodal trials, cortical or subcortical by the code's classify_region.

In [ ]:
READINGS = {"a": (0.0, 0.5, 0.010), "b": (0.0, 0.5, 0.005), "c": (0.5, 1.0, 0.005)}
mi_rows = []
MI50 = {}
for s in CAT_SESS:
    u, tr, trains, meta = DATA[s]
    idx = np.where(u.keep & u.sub_cort.notna())[0]
    for a in estim.AMPS:
        st = estim.condition_trials(tr, "cathodal", a).start_time.to_numpy()
        for i in idx:
            row = dict(session=s, unit=i, amp=a, Region=u.sub_cort[i], region=u.region[i], distance=u.distance[i])
            for k, (t0, t1, bw) in READINGS.items():
                mi = estim.modulation_index(trains[i], st, t0, t1, bw)
                row[f"auc_{k}"] = np.trapezoid(np.abs(mi))
            mi_rows.append(row)
MIDF = pd.DataFrame(mi_rows)
print("Units:", MIDF.groupby(["Region"]).unit.size().div(4).to_dict(), "per amplitude")
for k in READINGS:
    col = f"auc_{k}"
    print(f"\nReading {k}: mean +- SD")
    print(MIDF.groupby(["Region", "amp"])[col].agg(lambda x: f"{x.mean():.2f} +- {x.std():.2f}").unstack().to_string())
    t = anova_lm(smf.ols(f"{col} ~ amp * C(Region)", data=MIDF).fit(), typ=2)
    print("   ANOVA:", {i: f"F = {t.loc[i, 'F']:.2f}, p = {t.loc[i, 'PR(>F)']:.2e}" for i in t.index[:-1]})
    for a in estim.AMPS:
        c = MIDF[(MIDF.amp == a) & (MIDF.Region == "Cortical")][col]
        sc = MIDF[(MIDF.amp == a) & (MIDF.Region == "Subcortical")][col]
        print(f"   {a} uA: Student p = {stats.ttest_ind(c, sc).pvalue:.2e}, Welch p = {stats.ttest_ind(c, sc, equal_var=False).pvalue:.2e}")
    slope = {r: np.polyfit(g.amp, g[col], 1)[0] for r, g in MIDF.groupby("Region")}
    print("   slope on amplitude per region:", {r: round(v, 4) for r, v in slope.items()})
print("By recording, reading a, cortical mean at 5 and 100 uA:",
      MIDF[MIDF.Region == "Cortical"].groupby(["session", "amp"]).auc_a.mean().unstack()[[5, 100]].round(2).to_dict("index"))
print("\nExpected cortical 12.60 (5 uA), 20.21 (100 uA); subcortical 12.27, 13.48 (10%)")

**Result: partially reproduced.** Under the code's reading (a) the magnitudes are 12.60, 13.37,
15.11 and 20.21 (cortical) and 12.27, 12.60, 12.08 and 13.48 (subcortical), the paper's values, with
region F = 106.27, amplitude F = 487.36 and interaction F = 90.88 (all p < 1e-20) and cortical against
subcortical Student p = 4.65e-09 at 50 uA and 5.50e-29 at 100 uA. Under the legend's 5 ms bins (b) the
effects hold but the 50 uA comparison has p = 1.10e-02. Under the Methods' window of 500 to 1000 ms (c)
there is no region effect (p = 0.457) and the amplitude effect has p = 1.06e-03; the criterion asks for
all three readings.

## fig6d-hva-subcortical

> At 100 uA higher visual areas, beyond the reach of direct activation, are more modulated than
> subcortical units (15.82 against 13.48) although both lie at similar distances from the
> stimulation site.

Groups by the Figure 6 legend: V1 (primary visual area), HVA (LM, AM, PM), subcortical (the code's
list). Variant: the code's HVA (any CCF name with "layer" and not "primary"). Tukey HSD over the three
at 100 uA cathodal (magnitude under reading (a)); Mann-Whitney test of distances, HVA against
subcortical, over the good stable units of all nine recordings (as the code, cells 40 and 41).

In [ ]:
def hva_legend(r):
    return any(k in r for k in ["Lateral visual area", "Anteromedial visual area", "posteromedial visual area"])
def hva_code(r):
    return ("layer" in r) and ("primary" not in r.lower())
allkeep = pd.concat([DATA[s][0].assign(session=s) for s in estim.PAPER_SESSIONS])
allkeep = allkeep[allkeep.keep]
for name, fn in [("legend", hva_legend), ("code", hva_code)]:
    d = MIDF[MIDF.amp == 100].copy()
    d["group"] = np.where(d.region.str.contains("Primary visual area"), "V1",
                          np.where(d.region.map(fn), "HVA", np.where(d.Region == "Subcortical", "Subcortical", "other")))
    d = d[d.group != "other"]
    print(f"HVA by the {name}: n", d.group.value_counts().to_dict(), "| mean +- SD",
          {g: f"{x.mean():.2f} +- {x.std():.2f}" for g, x in d.groupby("group").auc_a})
    tk = pairwise_tukeyhsd(d.auc_a, d.group)
    print(pd.DataFrame(tk.summary().data[1:], columns=tk.summary().data[0]).to_string(index=False))
    k = allkeep.copy()
    k["group"] = np.where(k.region.map(fn) & ~k.region.str.contains("Primary visual area"), "HVA",
                          np.where(k.sub_cort == "Subcortical", "Subcortical", "other"))
    h, sc = k[k.group == "HVA"].distance, k[k.group == "Subcortical"].distance
    print(f"   distance HVA median {h.median():.0f} um (n={len(h)}), subcortical {sc.median():.0f} um (n={len(sc)}), Mann-Whitney p = {stats.mannwhitneyu(h, sc).pvalue:.3g}\n")
print("Expected HVA 15.82, subcortical 13.48 (10%); paper Tukey p = 0.004; Mann-Whitney p = 0.511")

**Result: not reproduced.** With the legend's groups (LM, AM and PM) HVA units are modulated at
13.91 and subcortical units at 13.48 at 100 uA (Tukey p = 0.902); the distance test agrees with the
paper (p = 0.577). With the code's broader HVA group (any layered name outside V1, 349 units) the
values are 15.82 and 13.68 and Tukey p = 0.0202, but the distances then differ (p = 5.92e-08).

## fig6e-circuit-responders

> The proportion of cortical units in the circuit response grows with amplitude (0.14, 0.21, 0.52,
> 0.79); subcortical units respond less but also increase (0.08, 0.09, 0.17, 0.35).

ZETA test (zetapy 2.7.2 `getZeta`, 300 ms after onset, seeded per unit and condition), p < 0.05.
As the code (figure7.ipynb cells 1 and 3 to 8), the trials are all those of the condition's first
run (here restricted to single 100 us pulses), the units good and stable, and the regions the code's
map_region_to_group. The tests are run for the cathodal conditions, which give the claim's values
and the cathodal model `Value ~ Amplitude * C(Region)` (type II), whose F values the Results also
print. Running all three polarities for the code's full model `Value ~ Amplitude * C(Polarity) *
C(Region)` would take about 20 more minutes; the full model is shown with the paper's cached
p-values (outside input, a check only).

In [ ]:
jobs, keys = [], []
for si, s in enumerate(estim.PAPER_SESSIONS):
    u, tr, trains, meta = DATA[s]
    idx = np.where(u.keep & u.region_group.notna())[0]
    for ai, a in enumerate(estim.AMPS):
        c = estim.condition_trials(tr, "cathodal", a, n=None)
        if len(c) == 0:
            continue
        run = c.run.iloc[0]
        st = tr[tr.run == run].start_time.to_numpy()
        for i in idx:
            jobs.append((trains[i], st, int(1_000_000 * si + 10_000 * ai + i)))
            keys.append((s, i, "cathodal", a, int(run)))
t = time.time()
ZP = estim.zeta_pvalues(jobs)
print(f"{len(jobs)} ZETA tests in {time.time() - t:.0f} s; failed: {int(np.isnan(ZP).sum())}")
Z = pd.DataFrame(keys, columns=["session", "unit", "pol", "amp", "run"]).assign(p=ZP)
Z = Z.join(pd.concat([DATA[s][0].assign(session=s).set_index(["session", "unit"], drop=False)[["region_group", "sub_cort", "distance"]]
                      for s in estim.PAPER_SESSIONS]), on=["session", "unit"])
Z["Region"] = np.where(Z.region_group.str.contains("Area"), "cortical", np.where(Z.region_group == "subcortical", "subcortical", None))
Z["sig"] = Z.p < 0.05
props = Z[Z.Region.notna()].groupby(["Region", "amp", "session"]).sig.mean().rename("Value").reset_index()
print(props.groupby(["Region", "amp"]).Value.agg(lambda x: f"{x.mean():.3f} +- {x.std():.3f} (n={len(x)})").unstack().to_string())
print("By recording, cortical:", props[props.Region == "cortical"].pivot(index="session", columns="amp", values="Value").round(2).to_dict("index"))
t = anova_lm(smf.ols("Value ~ amp * C(Region)", data=props).fit(), typ=2)
print("Cathodal model:", {i: f"F = {t.loc[i, 'F']:.2f}, p = {t.loc[i, 'PR(>F)']:.2e}" for i in t.index[:-1]})

# The paper's cached ZETA p-values (outside input, a check only)
CZ = {s: estim.code_zeta(s) for s in estim.PAPER_SESSIONS}
zc = Z.assign(p_code=[CZ[s][r][i] if r in CZ[s] else np.nan for s, r, i in zip(Z.session, Z.run, Z.unit)])
zc = zc[zc.p_code.notna()]
print(f"\nCached p-values found for {len(zc)} of {len(Z)} tests; same decision (p < 0.05) for {((zc.p < 0.05) == (zc.p_code < 0.05)).mean():.3f}; "
      f"responders here {(zc.p < 0.05).mean():.3f}, in the cache {(zc.p_code < 0.05).mean():.3f}; Pearson r of p {stats.pearsonr(zc.p, zc.p_code)[0]:.3f}")
rows = []
for s in estim.PAPER_SESSIONS:
    u, tr = DATA[s][0], DATA[s][1]
    idx = np.where(u.keep & u.region_group.notna())[0]
    reg = np.where(u.region_group.str.contains("Area", na=False), "cortical", np.where(u.region_group == "subcortical", "subcortical", None))
    for pol in estim.POLARITIES:
        for a in estim.AMPS:
            c = estim.condition_trials(tr, pol, a, n=None)
            if len(c) == 0 or int(c.run.iloc[0]) not in CZ[s]:
                continue
            pc = CZ[s][int(c.run.iloc[0])]
            for R in ["cortical", "subcortical"]:
                ii = idx[reg[idx] == R]
                rows.append(dict(session=s, pol=pol, amp=a, Region=R, Value=np.mean(pc[ii] < 0.05)))
cp = pd.DataFrame(rows)
print("Cached p-values, cathodal:", cp[cp.pol == "cathodal"].groupby(["Region", "amp"]).Value.mean().round(2).unstack().to_dict("index"))
t = anova_lm(smf.ols("Value ~ amp * C(pol) * C(Region)", data=cp).fit(), typ=2)
print("Cached p-values, full model:", {i: f"F = {t.loc[i, 'F']:.2f}, p = {t.loc[i, 'PR(>F)']:.2e}" for i in t.index[:-1]})
print("Units with 'posteromedial visual area' names, which the code's grouping drops:",
      int(allkeep.region.str.contains("posteromedial visual area").sum()))
print("Expected cortical 0.14, 0.21, 0.52, 0.79; subcortical 0.08 (5 uA), 0.35 (100 uA) (10%); paper F amplitude 175.18, region 81.50 (full model)")

**Result: partially reproduced.** With zetapy's getZeta the proportion of cortical circuit
responders is 0.125, 0.209, 0.520 and 0.785 and of subcortical ones 0.089, 0.094, 0.164 and 0.360 at
5 to 100 uA, rising with amplitude and above one half for cortex at 50 and 100 uA; the cathodal model
gives amplitude F = 89.48 (p = 6.84e-13) and region F = 36.59 (p = 1.61e-07). Two values are just
outside 10% (cortical 5 uA 0.125 against 0.14, subcortical 5 uA 0.089 against 0.08). The recomputed
p-values agree with the paper's cached ones (Pearson r = 0.992, same decision for 0.970 of 7980
tests), and the cached ones give the paper's 0.14, 0.21, 0.52, 0.79 and 0.08 to 0.35 on the dataset's
units, with amplitude F = 172.99 and region F = 85.99 in the code's full model.

## fig6f-no-propagating-wave

> Response timing shows no propagating wave: time to peak inhibition and to peak rebound excitation
> do not correlate with distance from the stimulation site.

As the code (figure7.ipynb cells 20 to 23): cortical good stable units that are ZETA responders to
50 uA cathodal pulses; MI in 5 ms bins over 0 to 500 ms (first 75 trials), Gaussian smoothing
(sigma 2.5 ms), the most prominent peak after 20 ms (prominence 0.2), time by the local centre of
mass. The distance is the code's: Euclidean distance to the first trial's cathodal contact, signed
negative where the unit is dorsal to it (the code takes the anodal contact for jlh32; both shown).
jlh39 and jlh49v1, which the code leaves out, have no cathodal pulses, so the two readings are the
same.

In [ ]:
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
BIN = 0.005
def pick_peaks(resp, sign):
    sig = resp if sign > 0 else -resp
    out = np.full(len(resp), np.nan)
    minlat = int(np.floor(0.02 / BIN))
    for k, y0 in enumerate(sig):
        y = y0.copy()
        y[:minlat] = -np.inf
        pk, props = find_peaks(y, prominence=0.2)
        if len(pk) == 0:
            idx = int(np.nanargmax(y[minlat:]) + minlat)
        else:
            idx = int(pk[np.argmax(props["prominences"])])
        w = np.maximum(y0, 0)
        lo, hi = max(0, idx - 1), min(len(w), idx + 2)
        out[k] = (np.sum(np.arange(lo, hi) * w[lo:hi]) / w[lo:hi].sum() if w[lo:hi].sum() > 0 else idx) * BIN
    return out

z50 = Z[(Z.pol == "cathodal") & (Z.amp == 50) & Z.sig]
for variant in ["code (jlh32 anodal contact)", "cathodal contact for all"]:
    mis, dists, sess = [], [], []
    for s in CAT_SESS:
        u, tr, trains, meta = DATA[s]
        st = estim.condition_trials(tr, "cathodal", 50).start_time.to_numpy()
        f0 = tr.iloc[0]
        stim = np.array([f0.neg_ap, f0.neg_dv, f0.neg_ml])
        if s == "jlh32" and variant.startswith("code"):
            f = estim.open_asset([a for a in ASSETS if a["session"] == s][0]["asset_id"])
            order = np.argsort(f["intervals/trials/start_time"][:], kind="stable")
            stim = f["intervals/trials/contact_positive_coords"][:][order][0]
            f.close()
        ids = z50[z50.session == s].unit.to_numpy()
        ids = [i for i in ids if "area" in u.region[i].lower()]
        for i in ids:
            mis.append(estim.modulation_index(trains[i], st, 0, 0.5, BIN))
            xyz = np.array([u.ap[i], u.dv[i], u.ml[i]])
            dists.append(np.linalg.norm(xyz - stim) * (-1 if xyz[1] < stim[1] else 1))
            sess.append(s)
    sm = gaussian_filter1d(np.array(mis), sigma=0.0025 / BIN, axis=1, mode="nearest")
    t_exc, t_inh = pick_peaks(sm, 1), pick_peaks(sm, -1)
    d = np.array(dists)
    print(f"{variant}: units {len(d)} by recording {pd.Series(sess).value_counts().to_dict()}; signed distance {d.min():.0f} to {d.max():.0f} um")
    for name, tt in [("excitatory", t_exc), ("inhibitory", t_inh)]:
        ok = np.isfinite(tt) & np.isfinite(d)
        r, p = stats.pearsonr(d[ok], tt[ok])
        print(f"   time to peak {name}: median {np.median(tt[ok]) * 1000:.0f} ms, Pearson r = {r:.3f}, p = {p:.3f}, n = {ok.sum()}")
print("Paper: Pearson r = -0.03 (p = 0.51) and 0.09 (p = 0.64)")

**Result: reproduced.** Among 614 cortical circuit responders at 50 uA, time to peak excitation
(Pearson r = -0.001, p = 0.987) and to peak inhibition (r = 0.047, p = 0.244) do not correlate with
signed distance, with either contact for jlh32.

## figs9-state-drp

> Direct response probability is slightly but significantly higher on high than on low
> cortical-state trials at every cathodal amplitude.

Trials (first 75 cathodal) split at the median of the population spike count in the 500 ms before
onset (high: above the median); DRP or nDRP per unit within each half; paired Wilcoxon over units, FDR
over amplitudes. Readings: population (a) all good stable units, (b) cortical ones; measure (i) DRP,
(ii) nDRP; units (x) all good stable units, (y) the 99 direct responders.

In [ ]:
rows = []
for s in CAT_SESS:
    u, tr, trains, meta = DATA[s]
    idx = np.where(u.keep)[0]
    for popname, pidx in [("a", idx), ("b", np.where(u.keep & (u.sub_cort == "Cortical"))[0])]:
        pop = np.sort(np.concatenate([trains[i] for i in pidx]))
        for a in estim.AMPS:
            st = estim.condition_trials(tr, "cathodal", a).start_time.to_numpy()
            cnt = np.searchsorted(pop, st) - np.searchsorted(pop, st - 0.5)
            hi = cnt > np.median(cnt)
            for i in idx:
                for state, m in [("high", hi), ("low", ~hi)]:
                    p, pb, n = estim.spike_prob(trains[i], st[m])
                    rows.append((s, i, popname, a, state, p, n))
ST = pd.DataFrame(rows, columns=["session", "unit", "pop", "amp", "state", "drp", "ndrp"])
resp_set = set(map(tuple, R99.index))
for popname in ["a", "b"]:
    for meas in ["drp", "ndrp"]:
        for uset in ["x", "y"]:
            d = ST[ST["pop"] == popname]
            if uset == "y":
                d = d[[k in resp_set for k in zip(d.session, d.unit)]]
            w = d.pivot_table(index=["session", "unit", "amp"], columns="state", values=meas).dropna().reset_index()
            res, ps = {}, []
            for a in estim.AMPS:
                x = w[w.amp == a]
                ps.append(stats.wilcoxon(x.high, x.low).pvalue if (x.high != x.low).any() else 1.0)
                res[a] = f"low {x.low.mean():.4f}, high {x.high.mean():.4f}"
            adj = multipletests(ps, method="fdr_bh")[1]
            print(f"population {popname}, {meas}, units {uset} (n={w.groupby('amp').size().iloc[0]}):", res, "| FDR p", np.round(adj, 4).tolist(),
                  "| high above low at", [a for a in estim.AMPS if w[w.amp == a].high.mean() > w[w.amp == a].low.mean()])
print("Expected DRP low 0.051, high 0.056 at 5 uA; low 0.104, high 0.107 at 100 uA (10%); paper p < 0.001, < 0.001, 0.010, 0.030")

**Result: partially reproduced.** With all good stable units, the whole population and DRP
(readings a, i, x), DRP is higher on high-state trials at every amplitude (0.0124 against 0.0120 at
5 uA, 0.0165 against 0.0159 at 100 uA; FDR p 0.0016 or less); the cortical population gives the same
(FDR p up to 0.0156). The values are a quarter of the paper's (0.051 and 0.056 at 5 uA). Among the 99
direct responders no difference is significant (FDR p 0.3008 to 0.9463), and with nDRP over all
units the low state is higher at every amplitude (FDR p 0.0336 or less).

## fig6g-dimensionality-collapse

> After a pulse the population's dimensionality drops (participation ratio and effective
> dimensionality fall), with a brief rise in effective dimensionality within 20 ms at 5, 25 and
> 50 uA.

Per recording, cortical good stable units (CCF name with "area"; the code's "layer" filter as a
variant), first 75 cathodal trials, spike counts in bins from -200 to +500 ms; per bin the
participation ratio of the neuron covariance across trials and the number of components for 95% of
the variance; each recording's trace divided by its median over pre-stimulus bins, then averaged over
recordings. Epochs as the code: middle 12 to 150 ms. Bins 2 ms (Methods, code) and 5 ms (legend).

In [ ]:
DIM = {}
for unit_filter in ["area", "layer"]:
    for bw in [0.002, 0.005]:
        edges = np.round(np.arange(-0.2, 0.5 + bw / 2, bw), 6)
        ctr = edges[:-1] + bw / 2
        res = {}
        for a in estim.AMPS:
            prs, eds = [], []
            for s in CAT_SESS:
                u, tr, trains, meta = DATA[s]
                idx = np.where(u.keep & u.region.str.contains(unit_filter))[0]
                st = estim.condition_trials(tr, "cathodal", a).start_time.to_numpy()
                C = np.stack([estim.counts_matrix(trains[i], st, edges) for i in idx], axis=2)  # trials x bins x units
                pr, ed = np.array([estim.participation_ratio(C[:, b, :]) for b in range(C.shape[1])]).T
                pre = ctr < 0
                prs.append(pr / np.nanmedian(pr[pre]))
                eds.append(ed / np.nanmedian(ed[pre]))
            res[a] = (np.nanmean(prs, 0), np.nanmean(eds, 0))
        DIM[(unit_filter, bw)] = (ctr, res)
        mid = (ctr >= 0.012) & (ctr <= 0.150)
        early = (ctr >= 0) & (ctr <= 0.020)
        print(f"units '{unit_filter}', {bw * 1000:.0f} ms bins:")
        for a in estim.AMPS:
            pr, ed = res[a]
            print(f"   {a} uA: middle-epoch PR {np.nanmean(pr[mid]):.3f}, ED {np.nanmean(ed[mid]):.3f} | 0-20 ms ED mean {np.nanmean(ed[early]):.3f}, max {np.nanmax(ed[early]):.3f}"
                  f" | late 170-300 ms PR {np.nanmean(pr[(ctr >= 0.17) & (ctr <= 0.3)]):.3f}")
ctr, res = DIM[("area", 0.002)]
fig, ax = plt.subplots(1, 2, figsize=(9, 2.5))
for a in estim.AMPS:
    ax[0].plot(ctr * 1000, res[a][0], label=f"{a} uA"); ax[1].plot(ctr * 1000, res[a][1])
ax[0].set_ylabel("normalised PR"); ax[1].set_ylabel("normalised ED"); ax[0].legend(fontsize=6)
for x in ax: x.set_xlabel("time from onset (ms)"); x.axhline(1, color="k", lw=0.5)
plt.tight_layout(); plt.show()
print("Plotted ranges: PR %.2f to %.2f, ED %.2f to %.2f" % (min(np.nanmin(res[a][0]) for a in estim.AMPS), max(np.nanmax(res[a][0]) for a in estim.AMPS),
                                                       min(np.nanmin(res[a][1]) for a in estim.AMPS), max(np.nanmax(res[a][1]) for a in estim.AMPS)))

**Result: reproduced.** With 2 ms bins, the normalised participation ratio in 12 to 150 ms is
0.982, 0.951, 0.765 and 0.459 and the effective dimensionality 0.986, 0.961, 0.778 and 0.476 at 5 to
100 uA, all below 1. Within 0 to 20 ms the largest bin of effective dimensionality is 1.024, 1.087 and
1.094 at 5, 25 and 50 uA (reading b; also 1.043 at 100 uA), while its mean over the window is below 1
(reading a). With 5 ms bins the largest bin exceeds 1 only at 25 and 50 uA.

## Summary

| Claim | Outcome |
|---|---|
| fig4d-latency-trough | reproduced |
| figs3cd-gmm-window | reproduced |
| fig4h-extent-constant | reproduced |
| fig4h-distance-decreases | partially reproduced |
| fig4h-ks-distinct | not reproduced |
| fig4j-drp-increases | reproduced |
| figs3e-early-windows | partially reproduced |
| fig5bc-amplitude-clusters | reproduced |
| fig5ef-polarity-clusters | partially reproduced |
| fig5-fsrs-proportion | partially reproduced (assumption: the code's FS/RS classes) |
| fig5j-fs-closer | reproduced (assumption: the code's FS/RS classes) |
| figs6b-fsrs-early-firing | reproduced (assumption: the code's FS/RS classes) |
| fig6c-modulation-cortex | partially reproduced |
| fig6d-hva-subcortical | not reproduced |
| fig6e-circuit-responders | partially reproduced |
| fig6f-no-propagating-wave | reproduced |
| figs9-state-drp | partially reproduced |
| fig6g-dimensionality-collapse | reproduced |
| fig1l, figs1b, fig2g (2), fig2-extent-polarity, fig2h, figs2gh, fig3c, fig3e, fig3fg, fig3h (11 claims) | blocked: the raw AP band is not in the dandiset |
| fig4f-responders-within-ep, fig4i-proportion-within-ep, fig5i-fs-within-ep | blocked: need the evoked-potential boundaries |
| fig7d, fig7e, fig7g, fig7h (2), figs10c, fig7i, fig7j, no-learning-across-sessions, figs10e (10 claims) | blocked: the behaviour is not in the dandiset |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

**How the analysis differs from the paper's.** Units, trials, the stability filter and the direct
response follow the paper and its code; the unit counts per condition match the code's intermediate
table. Distances use the dataset's coordinates; the paper's code uses its own, which differ for the
stimulation contacts of jlh31 and jlh32, and the distance claims are also shown with the code's
coordinates, which give the paper's numbers exactly. The signed distance needs channel coordinates
that the dandiset does not give; they are fitted from the units' coordinates. The FS and RS classes
follow the paper's code (a 0.4 ms duration threshold), where the Methods describe k-means; both are
shown. ZETA uses zetapy 2.7.2, whose `getZeta` the code calls, and only the cathodal conditions are
tested. The claims on cathodal pulses rest on seven recordings: two of the nine hold no cathodal
pulses.

**What was found about the dataset.** The dandiset holds the spikes, templates, CCF coordinates and
regions of the nine recordings the paper analyses, and a tenth (jlh41) without coordinates that the
paper does not use. The raw AP band, LFP and behaviour are not deposited, so 24 of the 42 claims
cannot be tested. The file of session jlh49v1 is filed under subject jlh48. The stimulation contact
coordinates of jlh31 and jlh32 differ from those the paper's code uses by 206 and 373 um, which
changes the outcome of the distance claims. The electrodes table holds placeholder locations and
coordinates. jlh33 has 60 trials per condition, where the Methods state 75. Neither FS/RS labels nor
the stability flags the paper uses are in the files; both are recomputed here.